## Lectura del dataset
Primero, leemos el dataset. Consideramos que es indispensable quitar las columnas que asignan un "orden" a los municipios y entidades federativas pues al no eliminarlas podemos generar que el algoritmo de clasificación busque patrones sobre el "orden" de esas entidades cuando realmente no existe. Podría suceder que por ejemplo encuentre relaciones estre la entidad federativa 23 con la 24, cuando no hay relación alguna. Así mismo, debemos de ser cuidadosos con las filas confidenciales, por lo que haremos una prueba eliminandolas.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Leer el CSV
df = pd.read_csv("data/data_secretariado.csv")
df.drop(["ID_VICTIMA", "CVE_ENT", "CVE_MUN"], axis=1, inplace=True)

# Convertir las fechas a tipo datetime
df["FECHA_NACIMIENTO"]   = pd.to_datetime(df["FECHA_NACIMIENTO"], errors="coerce")
df["FECHA_DESAPARICION"] = pd.to_datetime(df["FECHA_DESAPARICION"], errors="coerce")
df["FECHA_REGISTRO"] = pd.to_datetime(df["FECHA_REGISTRO"], errors="coerce")

# Calcular edad al momento de la desaparición (en años)
df["EDAD"] = (df["FECHA_DESAPARICION"] - df["FECHA_NACIMIENTO"]).dt.days / 365.25
df["DIAS_REPORTE"] = (df["FECHA_REGISTRO"] - df["FECHA_DESAPARICION"]).dt.days

df = df.drop(["FECHA_NACIMIENTO", "FECHA_DESAPARICION", "FECHA_REGISTRO"], axis=1)

df["EDAD"] = df["EDAD"].fillna(-1)
df["DIAS_REPORTE"] = df["DIAS_REPORTE"].fillna(-1)


# Partición en: 75% train, 25% test
train, test = train_test_split(df, test_size=0.25, random_state=42, stratify=df["ESTATUS_VICTIMA"])

# Verificar tamaños
print(f"Total:      {len(df)} filas")
print(f"Train (75%): {len(train)} filas")
print(f"Test (25%): {len(test)} filas")

Total:      133887 filas
Train (75%): 100415 filas
Test (25%): 33472 filas


/var/folders/m8/3v1lg6wd3g92fgsqwl_85ykh0000gn/T/ipykernel_14148/3223487789.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["FECHA_NACIMIENTO"]   = pd.to_datetime(df["FECHA_NACIMIENTO"], errors="coerce")
/var/folders/m8/3v1lg6wd3g92fgsqwl_85ykh0000gn/T/ipykernel_14148/3223487789.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["FECHA_DESAPARICION"] = pd.to_datetime(df["FECHA_DESAPARICION"], errors="coerce")
/var/folders/m8/3v1lg6wd3g92fgsqwl_85ykh0000gn/T/ipykernel_14148/3223487789.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["FECHA_REGISTRO"] = pd.

In [40]:
X_train = train.drop(columns="ESTATUS_VICTIMA")
y_train = train["ESTATUS_VICTIMA"]

X_test = test.drop(columns="ESTATUS_VICTIMA")
y_test = test["ESTATUS_VICTIMA"]

print(X_train.shape, y_train.shape)
print(X_train.head(10))

(100415, 6) (100415,)
                                           ORIGEN_REPORTE          SEXO  \
16312                         FISCALIA GENERAL DE NAYARIT  CONFIDENCIAL   
22485         FISCALIA GENERAL DEL ESTADO BAJA CALIFORNIA         MUJER   
34155                                              PORTAL  CONFIDENCIAL   
41677   COMISION LOCAL DE BUSQUEDA DE PERSONAS DEL EST...  CONFIDENCIAL   
38641                      FISCALIA GENERAL DE NUEVO LEON  CONFIDENCIAL   
69361                                              PORTAL  CONFIDENCIAL   
121047            FISCALIA GENERAL DEL ESTADO DE COAHUILA        HOMBRE   
14732           COMISION NACIONAL DE BUSQUEDA DE PERSONAS        HOMBRE   
96303                      FISCALIA GENERAL DE NUEVO LEON         MUJER   
68352                FISCALIA GENERAL DE ESTADO DE MEXICO         MUJER   

                 ENTIDAD       MUNICIPIO       EDAD  DIAS_REPORTE  
16312            NAYARIT    CONFIDENCIAL  -1.000000          -1.0  
22485    BAJA CA

## Modelo
Usaremos un modelo de RandomForest. Creemos que es un buen modelo pues queremos explorar arboles, pero tratando de explorar varias posibles relaciones entre atributos en lugar de asumir que todos tienen relacion entre si.

In [41]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.metrics import f1_score
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

SEED = 1

cat_ohe = ["SEXO", "ENTIDAD"]
cat_ord = ["MUNICIPIO", "ORIGEN_REPORTE"]
num_cols = ["EDAD", "DIAS_REPORTE"]

preprocessor = ColumnTransformer([
    ("ohe", OneHotEncoder(handle_unknown="ignore"), cat_ohe),
    ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_ord),
    ("num", "passthrough", num_cols),
])

pipe_rf = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=300, max_depth=None, random_state=SEED, n_jobs=-1
    ))
])


In [42]:
pipe_rf.fit(X_train, y_train)

print(f"Accuracy train: {pipe_rf.score(X_train, y_train):.4f}")
print(f"Accuracy test:  {pipe_rf.score(X_test, y_test):.4f}")

Accuracy train: 0.9894
Accuracy test:  0.9603


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = pipe_rf.predict(X_test)
print(classification_report(y_test, y_pred))
print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

               precision    recall  f1-score   support

 CONFIDENCIAL       1.00      1.00      1.00     12287
 DESAPARECIDA       0.96      0.97      0.97     19971
NO LOCALIZADA       0.44      0.33      0.38      1214

     accuracy                           0.96     33472
    macro avg       0.80      0.77      0.78     33472
 weighted avg       0.96      0.96      0.96     33472


Matriz de confusión:
[[12287     0     0]
 [    0 19455   516]
 [    0   812   402]]


Vamos a probar filtrar los datos confidenciales

In [44]:
df = pd.read_csv("data/data_secretariado.csv")
df.drop(["ID_VICTIMA", "CVE_ENT", "CVE_MUN"], axis=1, inplace=True)
df = df[df["ESTATUS_VICTIMA"] != "CONFIDENCIAL"]

# Convertir las fechas a tipo datetime
df["FECHA_NACIMIENTO"]   = pd.to_datetime(df["FECHA_NACIMIENTO"], errors="coerce")
df["FECHA_DESAPARICION"] = pd.to_datetime(df["FECHA_DESAPARICION"], errors="coerce")
df["FECHA_REGISTRO"] = pd.to_datetime(df["FECHA_REGISTRO"], errors="coerce")

# Calcular edad al momento de la desaparición (en años)
df["EDAD"] = (df["FECHA_DESAPARICION"] - df["FECHA_NACIMIENTO"]).dt.days / 365.25
df["DIAS_REPORTE"] = (df["FECHA_REGISTRO"] - df["FECHA_DESAPARICION"]).dt.days

df = df.drop(["FECHA_NACIMIENTO", "FECHA_DESAPARICION", "FECHA_REGISTRO"], axis=1)

df["EDAD"] = df["EDAD"].fillna(-1)
df["DIAS_REPORTE"] = df["DIAS_REPORTE"].fillna(-1)


# Partición en: 75% train, 25% test
train, test = train_test_split(df, test_size=0.25, random_state=42, stratify=df["ESTATUS_VICTIMA"])

# Verificar tamaños
print(f"Total:      {len(df)} filas")
print(f"Train (75%): {len(train)} filas")
print(f"Test (25%): {len(test)} filas")

Total:      84738 filas
Train (75%): 63553 filas
Test (25%): 21185 filas


In [45]:
X_train = train.drop(columns="ESTATUS_VICTIMA")
y_train = train["ESTATUS_VICTIMA"]

X_test = test.drop(columns="ESTATUS_VICTIMA")
y_test = test["ESTATUS_VICTIMA"]

print(X_train.shape, y_train.shape)
print(X_train.head(10))

(63553, 6) (63553,)
                                           ORIGEN_REPORTE    SEXO  \
109514                       FISCALIA GENERAL DE GUERRERO   MUJER   
19538   COMISION LOCAL DE BUSQUEDA DE PERSONAS DEL EST...   MUJER   
96181                        FISCALIA GENERAL DE GUERRERO  HOMBRE   
65452   PROCURADURIA GENERAL DE JUSTICIA DEL ESTADO DE...  HOMBRE   
131061                      FISCALIA GENERAL DE CHIHUAHUA  HOMBRE   
23166   COMISION LOCAL DE BUSQUEDA DE PERSONAS DEL EST...   MUJER   
37646                        FISCALIA GENERAL DE VERACRUZ  HOMBRE   
65766   PROCURADURIA GENERAL DE JUSTICIA DEL ESTADO DE...  HOMBRE   
534     PROCURADURIA GENERAL DE JUSTICIA DEL ESTADO DE...  HOMBRE   
19144   COMISION LOCAL DE BUSQUEDA DE PERSONAS DEL EST...   MUJER   

                 ENTIDAD                MUNICIPIO       EDAD  DIAS_REPORTE  
109514          GUERRERO             EDUARDO NERI  -1.000000           0.0  
19538   ESTADO DE MÉXICO      NAUCALPAN DE JUÁREZ  10.485969      

In [52]:
preprocessor = ColumnTransformer([
    ("ohe", OneHotEncoder(handle_unknown="ignore"), cat_ohe),
    ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_ord),
    ("num", "passthrough", num_cols),
])

pipe_rf = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=300, max_depth=None, random_state=SEED, n_jobs=-1, class_weight="balanced_subsample"
    ))
])

In [53]:
pipe_rf.fit(X_train, y_train)

print(f"Accuracy train: {pipe_rf.score(X_train, y_train):.4f}")
print(f"Accuracy test:  {pipe_rf.score(X_test, y_test):.4f}")

Accuracy train: 0.9628
Accuracy test:  0.9176


In [54]:
y_pred = pipe_rf.predict(X_test)
print(classification_report(y_test, y_pred))
print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

               precision    recall  f1-score   support

 DESAPARECIDA       0.97      0.95      0.96     19971
NO LOCALIZADA       0.34      0.46      0.39      1214

     accuracy                           0.92     21185
    macro avg       0.65      0.70      0.67     21185
 weighted avg       0.93      0.92      0.92     21185


Matriz de confusión:
[[18886  1085]
 [  661   553]]
